# 🛡️ GraphGuard: Colab Retraining Pipeline

This notebook trains the **GraphGuard (TempoStruct-GAN + GraphSAGE)** pipeline on GPU:
1. **Stage 1 (GAN)**: Trains the Temporal-Conditioned Generator and Edge Predictor.
2. **Stage 2 (Classifier)**: Synthesizes illicit nodes, injects time-consistent edges, and trains the downstream 2-layer GraphSAGE classifier to convergence.

---

### Step 1: Verify GPU Environment
> **Ensure GPU is selected**: Go to `Runtime` ➔ `Change runtime type` ➔ Select `T4 GPU`.

In [ ]:
!nvidia-smi

### Step 2: Unzip Project Files
Upload `graphguard_colab.zip` into the Colab file browser (left sidebar folder icon), then run this cell.

In [ ]:
import os

# If uploaded directly to Colab
if os.path.exists('/content/graphguard_colab.zip'):
    !unzip -q -o /content/graphguard_colab.zip -d /content/
    print("Unzipped graphguard_colab.zip in /content/")
# If stored in Google Drive
elif os.path.exists('/content/drive/MyDrive/graphguard_colab.zip'):
    !unzip -q -o /content/drive/MyDrive/graphguard_colab.zip -d /content/
    print("Unzipped from Google Drive!")
else:
    print("Please upload graphguard_colab.zip to the files pane on the left.")

### Step 3: Install Required Libraries

In [ ]:
%cd /content
!pip install -q scikit-learn matplotlib seaborn

### Step 4: Train Generator & Edge Predictor (TempoStruct-GAN)
Trains the temporal generator and edge predictor for 100 epochs with candidate subsampling to prevent OOM.

In [ ]:
!python scripts/02_train_generator.py

### Step 5: Train Downstream GraphSAGE Classifier
Balances training set with synthetic illicit nodes, injects edges, and trains GraphSAGE for 1000 epochs.

In [ ]:
!python scripts/03_train_classifier.py

### Step 6: View Training Curves & Final Test Metrics

In [ ]:
import glob, json
from IPython.display import Image, display

# Find latest classifier run
runs = sorted(glob.glob('/content/results/classifier_gnn/run_*'))
if runs:
    latest_run = runs[-1]
    print(f"Latest Run: {latest_run}\n")
    
    with open(f"{latest_run}/metrics.json") as f:
        metrics = json.load(f)
        print("="*40)
        print("FINAL TEST METRICS:")
        print("="*40)
        for k, v in metrics.items():
            print(f"{k:25s}: {v}")
        print("="*40)
        
    print("\n--- Validation F1 Curve ---")
    display(Image(f"{latest_run}/val_f1_curve.png"))
    print("\n--- Loss Curve ---")
    display(Image(f"{latest_run}/loss_curve.png"))

### Step 7: Zip Results & Models for Download

In [ ]:
!zip -r /content/graphguard_trained_results.zip results/ graphguard/models/*.pt
from google.colab import files
files.download('/content/graphguard_trained_results.zip')